# CEG 外部 baseline 产物 Colab Notebook

该 Notebook 在独立 Colab 冷启动会话中运行外部 baseline adapter。当前支持两类输入:

1. Tree-Ring / Gaussian Shading / Shallow Diffuse: 从仓库内置 `prompt_plan.json` 读取 prompt, 在当前 Colab GPU 会话中调用 SD3.5 Medium 生成外部 baseline 图像与统一 `baseline_observations.json`。
2. T2SMark: 读取前序 `colab_t2smark_baseline_outputs.ipynb` 已保存到 Google Drive 的 `external_baseline_inputs/t2smark/results.json`, 并从图像生成阶段归档恢复 `image_pairs.json` 进行 observation 适配。
3. RivaGAN / WAM / TrustMark: 从图像生成阶段归档恢复 `image_pairs.json`, 对 clean 图像执行补充表图像水印 baseline, 并在本 Notebook 内下载或触发加载所需权重。

冷启动顺序说明: 如果 Google Drive 的 CEG 目录为空, 必须先运行 `colab_pilot_image_generation_outputs.ipynb` 生成图像阶段归档；本 Notebook 不在 Drive 中运行代码, 只从 Drive 恢复前序阶段 zip 并在 `/content/ceg_runtime` 中运行。

本 Notebook 的最终产物是 Google Drive 中 `archives/external_baseline_outputs/<RUN_ID>.zip`, 供 `colab_paper_results_pipeline.ipynb` 在另一个独立 Colab 会话中恢复并导入论文结果包。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_CONTRACT_VERSION = "external_baseline_outputs_main_and_supplementary_cold_start_v1"
REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_BRANCH = ""
REPO_ROOT = Path("/content/CEG")
DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
LOCAL_RUNTIME_ROOT = Path("/content/ceg_runtime")

PROMPT_PLAN_PROFILE = "paper_main_probe"
IMAGE_GENERATION_RUN_ID = f"{PROMPT_PLAN_PROFILE}_image_generation_outputs"
RUN_ID = f"{IMAGE_GENERATION_RUN_ID}_external_baselines"
WORKSPACE = LOCAL_RUNTIME_ROOT / RUN_ID
RESET_LOCAL_RUNTIME_WORKSPACE = True

PROMPT_PLAN = REPO_ROOT / "prompts" / "prompt_plans" / f"{PROMPT_PLAN_PROFILE}_prompt_plan.json"
IMAGE_GENERATION_ARCHIVE = DRIVE_ROOT / "archives" / "image_generation_outputs" / f"{IMAGE_GENERATION_RUN_ID}.zip"
IMAGE_PAIRS = WORKSPACE / "inputs" / "images" / "image_pairs.json"

# 主表外部 baseline。Tree-Ring / Gaussian Shading / Shallow Diffuse 会直接从 prompt plan 生成各自图像和 observation。
SD35_BASELINE_METHODS = "tree_ring,gaussian_shading,shallow_diffuse"
RUN_SD35_BASELINES = True

# T2SMark 本体由 paper_workflow/baselines/colab_t2smark_baseline_outputs.ipynb 单独运行。
# 如果这里启用, 必须已经存在 Drive 中的 results.json 和图像生成阶段 image_pairs.json。
# 默认启用 T2SMark adapter。注意: 需要先运行 paper_workflow/baselines/colab_t2smark_baseline_outputs.ipynb,
# 并在 Drive 中准备 external_baseline_inputs/t2smark/results.json。
INCLUDE_T2SMARK_ADAPTER = True
T2SMARK_RESULTS = DRIVE_ROOT / "external_baseline_inputs" / "t2smark" / "results.json"

BASELINE_PLAN = WORKSPACE / "plans" / "external_baseline_plan.json"
SD35_BASELINE_PLAN = WORKSPACE / "plans" / "sd35_external_baseline_plan.json"
T2SMARK_BASELINE_PLAN = WORKSPACE / "plans" / "t2smark_baseline_plan.json"
BASELINE_OUTPUT_ROOT = WORKSPACE / "external_baselines"
SD35_OUTPUT_ROOT = BASELINE_OUTPUT_ROOT / "sd35_main_table_methods"
T2SMARK_OBSERVATION_OUTPUT = BASELINE_OUTPUT_ROOT / "t2smark" / "baseline_observations.json"

# 补充表图像水印 baseline。该类方法不从 prompt 直接生成图像, 而是读取图像生成阶段的 image_pairs.json。
# 因此在 Google Drive 为空目录的冷启动场景下, 需要先运行 colab_pilot_image_generation_outputs.ipynb 生成图像阶段归档。
RUN_SUPPLEMENTARY_IMAGE_WATERMARK_BASELINES = True
SUPPLEMENTARY_IMAGE_WATERMARK_METHODS = "rivagan_invisible_watermark,wam,trustmark"
SUPPLEMENTARY_BASELINE_PLAN = WORKSPACE / "plans" / "supplementary_image_watermark_baseline_plan.json"
SUPPLEMENTARY_OUTPUT_ROOT = BASELINE_OUTPUT_ROOT / "supplementary_table_methods"
SUPPLEMENTARY_REQUIRE_IMAGE_PAIRS = True

MODEL_CACHE_ROOT = WORKSPACE / "model_cache"
WAM_CHECKPOINT = MODEL_CACHE_ROOT / "watermark_anything" / "wam_mit.pth"
WAM_CHECKPOINT_URL = "https://dl.fbaipublicfiles.com/watermark_anything/wam_mit.pth"
DOWNLOAD_SUPPLEMENTARY_BASELINE_WEIGHTS = True
INSTALL_SUPPLEMENTARY_BASELINE_DEPENDENCIES = True
RIVAGAN_PAYLOAD = "CEG0"
TRUSTMARK_PAYLOAD = "cegmark"
TRUSTMARK_MODEL_TYPE = "Q"

INSTALL_EXTERNAL_BASELINE_DEPENDENCIES = True
RUN_EXTERNAL_BASELINES = True
REQUIRE_BASELINE_PASS = True
BASELINE_FORMAL_RESULT_CLAIM = False
BASELINE_EVIDENCE_PATHS = []
MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
TORCH_DTYPE = "float16"
HEIGHT = 512
WIDTH = 512
NUM_INFERENCE_STEPS = 28
NUM_INVERSION_STEPS = 28
GUIDANCE_SCALE = 7.0
ATTACK_FAMILIES = "brightness_contrast,gaussian_noise,rotate,resize,jpeg"
MAX_SAMPLES = None
REQUIRE_CUDA = True
HF_TOKEN_SECRET_NAMES = ["HF_TOKEN", "HUGGINGFACE_TOKEN"]


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print(f"非 Colab 环境或 Drive 已挂载: {exc}")

if not REPO_ROOT.exists():
    cmd = ["git", "clone"]
    if REPO_BRANCH:
        cmd += ["--branch", REPO_BRANCH]
    cmd += [REPO_URL, str(REPO_ROOT)]
    print("运行:", " ".join(cmd))
    subprocess.run(cmd, check=True)
else:
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--all", "--prune"], check=True)
    if REPO_BRANCH:
        subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO_ROOT)], check=True)
if INSTALL_EXTERNAL_BASELINE_DEPENDENCIES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "diffusers",
            "transformers",
            "accelerate",
            "safetensors",
            "Pillow",
            "scipy",
            "pycryptodome",
            "scikit-learn",
            "torchvision",
        ],
        check=True,
    )
if INSTALL_SUPPLEMENTARY_BASELINE_DEPENDENCIES:
    # 补充表 baseline 在独立 Colab 冷启动环境中不能依赖手工预装包。
    # 这里只安装第三方方法运行依赖, 不把任何权重或密钥写入仓库。
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "opencv-python-headless",
            "onnxruntime",
            "omegaconf",
            "matplotlib",
            "scikit-image",
        ],
        check=True,
    )
    rivagan_source = REPO_ROOT / "external_baselines" / "supplementary_table" / "rivagan_invisible_watermark" / "source"
    trustmark_repo_root = REPO_ROOT / "external_baselines" / "supplementary_table" / "trustmark" / "source"
    trustmark_source = trustmark_repo_root / "python"
    wam_source = REPO_ROOT / "external_baselines" / "supplementary_table" / "watermark_anything" / "source"

    def clone_or_update_external_source(repo_url: str, target: Path) -> None:
        # GitHub 上的 CEG 主仓库不提交第三方 source 快照。Colab 冷启动时在本地重新拉取第三方实现。
        target.parent.mkdir(parents=True, exist_ok=True)
        if (target / ".git").is_dir():
            subprocess.run(["git", "-C", str(target), "fetch", "--all", "--prune"], check=True)
            subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=True)
            return
        if target.exists() and any(target.iterdir()):
            raise RuntimeError(f"第三方 source 目录已存在但不是 git 仓库, 为避免误删请手动清理: {target}")
        subprocess.run(["git", "clone", repo_url, str(target)], check=True)

    clone_or_update_external_source("https://github.com/ShieldMnt/invisible-watermark.git", rivagan_source)
    clone_or_update_external_source("https://github.com/facebookresearch/watermark-anything.git", wam_source)
    clone_or_update_external_source("https://github.com/adobe/trustmark.git", trustmark_repo_root)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(rivagan_source)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(trustmark_source)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(wam_source / "requirements.txt")], check=True)

from paper_workflow.colab_utils.runtime import archive_directory_to_drive, extract_stage_archive, rewrite_image_pairs_for_restored_archive

# SD3.5 Medium 可能需要 Hugging Face token。这里只把 Colab secrets 写入运行时环境变量, 不落盘。
try:
    from google.colab import userdata
except Exception:
    userdata = None
for token_name in HF_TOKEN_SECRET_NAMES:
    if os.environ.get(token_name):
        continue
    if userdata is None:
        continue
    try:
        token_value = userdata.get(token_name)
    except Exception:
        token_value = None
    if token_value:
        os.environ[token_name] = token_value
        if token_name != "HF_TOKEN" and not os.environ.get("HF_TOKEN"):
            os.environ["HF_TOKEN"] = token_value
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
print("hf_token_configured =", bool(os.environ.get("HF_TOKEN")))

if WORKSPACE.exists() and RESET_LOCAL_RUNTIME_WORKSPACE:
    shutil.rmtree(WORKSPACE)
WORKSPACE.mkdir(parents=True, exist_ok=True)
(WORKSPACE / "plans").mkdir(parents=True, exist_ok=True)
BASELINE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
SUPPLEMENTARY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if IMAGE_GENERATION_ARCHIVE.is_file():
    extract_stage_archive(
        archive_zip_path=IMAGE_GENERATION_ARCHIVE,
        destination_root=WORKSPACE / "inputs" / "images",
        reset=True,
    )
    if IMAGE_PAIRS.is_file():
        rewrite_report = rewrite_image_pairs_for_restored_archive(
            image_pairs_path=IMAGE_PAIRS,
            image_output_root=WORKSPACE / "inputs" / "images",
        )
        print("image_pairs 路径重写报告 =")
        print(json.dumps(rewrite_report, ensure_ascii=False, indent=2))
        if rewrite_report.get("overall_decision") != "pass":
            raise RuntimeError("image_pairs 路径重写失败, external baseline 不能安全读取恢复后的图像。")
else:
    print("未找到图像生成阶段归档。Tree-Ring/Gaussian/Shallow 不依赖该归档, 但 T2SMark adapter 和补充表图像水印 baseline 需要 image_pairs.json:", IMAGE_GENERATION_ARCHIVE)

print("workspace =", WORKSPACE)
print("prompt_plan =", PROMPT_PLAN, "exists=", PROMPT_PLAN.exists())
print("image_generation_archive =", IMAGE_GENERATION_ARCHIVE, "exists=", IMAGE_GENERATION_ARCHIVE.exists())
print("image_pairs =", IMAGE_PAIRS, "exists=", IMAGE_PAIRS.exists())
print("t2smark_results =", T2SMARK_RESULTS, "exists=", T2SMARK_RESULTS.exists())
print("baseline_plan =", BASELINE_PLAN)
print("notebook_contract_version =", NOTEBOOK_CONTRACT_VERSION)
try:
    repo_commit = subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"], text=True).strip()
except Exception as exc:
    repo_commit = f"unknown: {exc}"
print("repo_commit =", repo_commit)


In [ ]:
# 2.5 补充表 baseline 权重准备
# 该 cell 面向“Colab 冷启动, Google Drive 为空目录”的场景。权重下载到本次 Colab 本地 WORKSPACE,
# 之后会随 external_baseline_outputs 归档保存回 Drive, 但不会作为下一次运行的必要输入。
import urllib.request

def download_if_missing(url: str, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and target.stat().st_size > 0:
        print("已存在:", target, "bytes=", target.stat().st_size)
        return
    print("下载:", url, "->", target)
    urllib.request.urlretrieve(url, target)
    print("完成:", target, "bytes=", target.stat().st_size)

if RUN_SUPPLEMENTARY_IMAGE_WATERMARK_BASELINES and DOWNLOAD_SUPPLEMENTARY_BASELINE_WEIGHTS:
    if "wam" in {item.strip() for item in SUPPLEMENTARY_IMAGE_WATERMARK_METHODS.split(",") if item.strip()}:
        download_if_missing(WAM_CHECKPOINT_URL, WAM_CHECKPOINT)

    # RivaGAN invisible-watermark 的 ONNX 权重应随第三方 source 一起存在。若不存在, 直接 fail-fast,
    # 避免静默退化为非 RivaGAN 方法。
    rivagan_model_dir = REPO_ROOT / "external_baselines" / "supplementary_table" / "rivagan_invisible_watermark" / "source" / "imwatermark"
    for model_name in ["rivagan_encoder.onnx", "rivagan_decoder.onnx"]:
        model_path = rivagan_model_dir / model_name
        if not model_path.is_file():
            raise FileNotFoundError(f"RivaGAN ONNX 权重不存在: {model_path}")
        print("RivaGAN 权重已准备:", model_path, "bytes=", model_path.stat().st_size)

    # TrustMark 权重由 TrustMark 源码在首次构造 TrustMark 对象时下载到其 package models 目录。
    # 这里不提前调用模型, 避免在 plan 构建阶段消耗显存；正式 adapter 执行时会触发下载。
    print("TrustMark 权重策略: adapter 首次运行时按 TrustMark 源码自动下载。")
else:
    print("未启用补充表 baseline 权重下载。")


In [ ]:
plans = []
if RUN_SD35_BASELINES:
    build_sd35_plan_cmd = [
        sys.executable,
        "scripts/build_sd35_external_adapter_baseline_plan.py",
        "--repo-root", str(REPO_ROOT),
        "--methods", SD35_BASELINE_METHODS,
        "--prompt-plan", str(PROMPT_PLAN),
        "--out", str(SD35_BASELINE_PLAN),
        "--output-root", str(SD35_OUTPUT_ROOT),
        "--working-directory", str(REPO_ROOT),
        "--timeout-seconds", "86400",
        "--model-id", MODEL_ID,
        "--torch-dtype", TORCH_DTYPE,
        "--height", str(HEIGHT),
        "--width", str(WIDTH),
        "--num-inference-steps", str(NUM_INFERENCE_STEPS),
        "--num-inversion-steps", str(NUM_INVERSION_STEPS),
        "--guidance-scale", str(GUIDANCE_SCALE),
        "--attack-families", ATTACK_FAMILIES,
    ]
    if MAX_SAMPLES is not None:
        build_sd35_plan_cmd += ["--max-samples", str(MAX_SAMPLES)]
    if REQUIRE_CUDA:
        build_sd35_plan_cmd.append("--require-cuda")
    print("运行:", " ".join(build_sd35_plan_cmd))
    subprocess.run(build_sd35_plan_cmd, check=True)
    plans.extend(json.loads(SD35_BASELINE_PLAN.read_text(encoding="utf-8-sig")))

if INCLUDE_T2SMARK_ADAPTER:
    if not IMAGE_PAIRS.is_file():
        raise FileNotFoundError(f"启用 T2SMark adapter 时必须存在 image_pairs.json: {IMAGE_PAIRS}")
    if not T2SMARK_RESULTS.is_file():
        raise FileNotFoundError(f"启用 T2SMark adapter 时必须先运行 T2SMark notebook 并保存 results.json: {T2SMARK_RESULTS}")
    build_t2smark_plan_cmd = [
        sys.executable,
        "scripts/build_t2smark_adapter_baseline_plan.py",
        "--repo-root", str(REPO_ROOT),
        "--image-pairs", str(IMAGE_PAIRS),
        "--t2smark-results", str(T2SMARK_RESULTS),
        "--out", str(T2SMARK_BASELINE_PLAN),
        "--observation-output", str(T2SMARK_OBSERVATION_OUTPUT),
        "--working-directory", str(REPO_ROOT),
        "--timeout-seconds", "7200",
    ]
    print("运行:", " ".join(build_t2smark_plan_cmd))
    subprocess.run(build_t2smark_plan_cmd, check=True)
    plans.extend(json.loads(T2SMARK_BASELINE_PLAN.read_text(encoding="utf-8-sig")))

if RUN_SUPPLEMENTARY_IMAGE_WATERMARK_BASELINES:
    if not IMAGE_PAIRS.is_file():
        message = (
            "启用补充表图像水印 baseline 时必须存在 image_pairs.json。"
            "在 Google Drive 为空目录的冷启动场景下, 请先运行 colab_pilot_image_generation_outputs.ipynb, "
            "生成 archives/image_generation_outputs/<run_id>.zip 后再运行本 Notebook。"
        )
        if SUPPLEMENTARY_REQUIRE_IMAGE_PAIRS:
            raise FileNotFoundError(f"{message} 缺失路径: {IMAGE_PAIRS}")
        print(message)
    else:
        build_supplementary_plan_cmd = [
            sys.executable,
            "scripts/build_supplementary_image_watermark_baseline_plan.py",
            "--repo-root", str(REPO_ROOT),
            "--methods", SUPPLEMENTARY_IMAGE_WATERMARK_METHODS,
            "--image-pairs", str(IMAGE_PAIRS),
            "--out", str(SUPPLEMENTARY_BASELINE_PLAN),
            "--output-root", str(SUPPLEMENTARY_OUTPUT_ROOT),
            "--working-directory", str(REPO_ROOT),
            "--timeout-seconds", "86400",
            "--attack-families", ATTACK_FAMILIES,
            "--rivagan-payload", RIVAGAN_PAYLOAD,
            "--trustmark-payload", TRUSTMARK_PAYLOAD,
            "--trustmark-model-type", TRUSTMARK_MODEL_TYPE,
        ]
        if WAM_CHECKPOINT.is_file():
            build_supplementary_plan_cmd += ["--wam-checkpoint", str(WAM_CHECKPOINT)]
        if MAX_SAMPLES is not None:
            build_supplementary_plan_cmd += ["--max-samples", str(MAX_SAMPLES)]
        print("运行:", " ".join(build_supplementary_plan_cmd))
        subprocess.run(build_supplementary_plan_cmd, check=True)
        plans.extend(json.loads(SUPPLEMENTARY_BASELINE_PLAN.read_text(encoding="utf-8-sig")))

if not plans:
    raise RuntimeError("没有生成任何 external baseline plan row。")
BASELINE_PLAN.write_text(json.dumps(plans, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("合并后的 baseline plan:", BASELINE_PLAN)
print(json.dumps({"command_count": len(plans), "baseline_ids": [row["baseline_id"] for row in plans]}, ensure_ascii=False, indent=2))

cmd = [
    sys.executable,
    "scripts/run_baseline_plan.py",
    "--plan", str(BASELINE_PLAN),
    "--out", str(BASELINE_OUTPUT_ROOT),
]
if REQUIRE_BASELINE_PASS:
    cmd.append("--require-pass")
if BASELINE_FORMAL_RESULT_CLAIM:
    cmd.append("--formal-result-claim")
for evidence_path in BASELINE_EVIDENCE_PATHS:
    cmd += ["--evidence-path", str(evidence_path)]
print("运行:", " ".join(cmd))
def print_baseline_failure_reports():
    report_paths = [
        BASELINE_OUTPUT_ROOT / "baseline_execution_manifest.json",
        BASELINE_OUTPUT_ROOT / "baseline_command_results.json",
        BASELINE_OUTPUT_ROOT / "baseline_command_plan_manifest.json",
        BASELINE_PLAN,
    ]
    for path in report_paths:
        print("报告:", path, "exists=", path.exists())
        if not path.is_file():
            continue
        text = path.read_text(encoding="utf-8", errors="replace")
        print(text[-12000:])

if RUN_EXTERNAL_BASELINES:
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError:
        print_baseline_failure_reports()
        raise
else:
    print("RUN_EXTERNAL_BASELINES = False, 仅打印命令。")


In [ ]:
archive_manifest = archive_directory_to_drive(
    source_root=BASELINE_OUTPUT_ROOT,
    drive_root=DRIVE_ROOT,
    archive_group="external_baseline_outputs",
    run_id=RUN_ID,
    archive_name=RUN_ID,
)
print("baseline_archive_manifest =")
print(archive_manifest)


In [ ]:
paths = {
    "prompt_plan": PROMPT_PLAN,
    "image_pairs": IMAGE_PAIRS,
    "t2smark_results": T2SMARK_RESULTS,
    "baseline_plan": BASELINE_PLAN,
    "sd35_plan": SD35_BASELINE_PLAN,
    "t2smark_plan": T2SMARK_BASELINE_PLAN,
    "supplementary_plan": SUPPLEMENTARY_BASELINE_PLAN,
    "wam_checkpoint": WAM_CHECKPOINT,
    "baseline_observations": BASELINE_OUTPUT_ROOT / "baseline_observations.json",
    "baseline_execution_manifest": BASELINE_OUTPUT_ROOT / "baseline_execution_manifest.json",
    "drive_archive": DRIVE_ROOT / "archives" / "external_baseline_outputs" / f"{RUN_ID}.zip",
}
for name, item_path in paths.items():
    print(name, "=", item_path, "exists=", item_path.exists())
if (BASELINE_OUTPUT_ROOT / "baseline_observations.json").is_file():
    rows = json.loads((BASELINE_OUTPUT_ROOT / "baseline_observations.json").read_text(encoding="utf-8-sig"))
    print("baseline_observation_count =", len(rows))
    print("baseline_ids =", sorted({row.get("baseline_id") for row in rows}))
